<a href="https://colab.research.google.com/github/phuongtvyz/LaDe-Spatiotemporal-Clustering-Thesis/blob/main/LaDe_D_Data_Audit_For_Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

!pip -q install pandas numpy matplotlib huggingface_hub

from pathlib import Path
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import json

BASE_DIR = Path("/content/LaDe_D_Research")
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "results"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "Cainiao-AI/LaDe"
CITIES = ["sh", "hz", "cq", "jl", "yt"]

FILES = {}

for city in CITIES:
    filename = f"delivery_{city}.csv"
    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=f"delivery/{filename}",
        local_dir=str(BASE_DIR / "source")
    )
    FILES[city] = Path(path)
    print(city, Path(path).stat().st_size, "bytes")

print("Download complete:", len(FILES), "files")


schema_rows = []

for city, path in FILES.items():
    sample = pd.read_csv(path, nrows=5)

    schema_rows.append({
        "city": city,
        "file": path.name,
        "file_size_MB": round(path.stat().st_size / 1024**2, 2),
        "columns": ", ".join(sample.columns)
    })

    print(f"\nCITY: {city}")
    print("Columns:", list(sample.columns))
    print("Data types:")
    print(sample.dtypes.to_string())
    display(sample.head(3))

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

schema_df.to_csv(
    OUT_DIR / "01_schema.csv",
    index=False
)
audit_rows = []
missing_rows = []

for city, path in FILES.items():
    print("Auditing:", city)

    df = pd.read_csv(path, low_memory=False)

    audit_rows.append({
        "city": city,
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "duplicate_full_rows": int(df.duplicated().sum())
    })

    for col in df.columns:
        missing_rows.append({
            "city": city,
            "column": col,
            "missing_count": int(df[col].isna().sum()),
            "missing_pct": round(
                df[col].isna().mean() * 100, 4
            )
        })

    del df

audit_df = pd.DataFrame(audit_rows)
missing_df = pd.DataFrame(missing_rows)

display(audit_df)
display(missing_df)

audit_df.to_csv(
    OUT_DIR / "02_record_audit.csv",
    index=False
)
missing_df.to_csv(
    OUT_DIR / "03_missing_audit.csv",
    index=False
)

import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/LaDe_D_Phase1_Results",
    "zip",
    root_dir=str(OUT_DIR)
)

print("Results saved:", zip_path)
files.download(zip_path)

# ============================================================
# CELL 5 - KIỂM TRA REGION, NGÀY, THỜI GIAN VÀ TỌA ĐỘ
# LaDe-D | Nguồn công bố: 01/05/2022 - 31/10/2022
# Không chạy thuật toán phân cụm
# ============================================================

from collections import Counter, defaultdict
import pandas as pd
import numpy as np

START_DATE = pd.Timestamp("2022-05-01")
END_DATE = pd.Timestamp("2022-10-31")
CHUNK_SIZE = 200_000

# Các biến để Cell 6 tiếp tục sử dụng
daily_counts = Counter()           # Theo ngày ds gốc
daily_delivery_counts = Counter()  # Theo ngày hoàn tất thực tế
aoi_counts = Counter()
hour_counts = Counter()
aoi_regions = defaultdict(set)
city_region_ids = defaultdict(set)

quality_rows = []

def parse_event_time(series):
    """Chuyển MM-DD HH:MM:SS thành datetime năm 2022."""
    text = series.astype("string").str.strip()

    return pd.to_datetime(
        "2022-" + text,
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

def parse_ds(series):
    """Chuyển ds dạng 604, 907, 1030 thành ngày năm 2022."""
    raw = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(4)
    )

    date = pd.to_datetime(
        "2022-" + raw.str[:2] + "-" + raw.str[2:4],
        format="%Y-%m-%d",
        errors="coerce"
    )

    return date

for city_code, path in FILES.items():

    print(f"\nĐang kiểm tra {city_code.upper()}...")

    q = Counter()
    seen_orders = set()
    city_names = set()
    days_seen = set()

    accept_min = None
    accept_max = None
    delivery_min = None
    delivery_max = None

    for ch in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False,
        dtype={
            "order_id": "string",
            "region_id": "string",
            "aoi_id": "string",
            "ds": "string"
        }
    ):

        q["total_rows"] += len(ch)
        city_names.update(
            ch["city"].dropna().astype(str).unique()
        )

        # 1. Kiểm tra trùng mã đơn hàng
        ids = ch["order_id"].dropna().str.strip()

        q["duplicate_order_ids"] += (
            int(ids.duplicated().sum()) +
            int(ids.drop_duplicates().isin(seen_orders).sum())
        )

        seen_orders.update(ids.unique().tolist())

        # 2. Kiểm tra mã khu vực
        region = ch["region_id"].str.strip()
        aoi = ch["aoi_id"].str.strip()

        city_region_ids[city_code].update(
            region.dropna().unique().tolist()
        )

        # 3. Chuyển đổi ngày tháng theo năm 2022
        ds_date = parse_ds(ch["ds"])
        accept = parse_event_time(ch["accept_time"])
        delivery = parse_event_time(ch["delivery_time"])

        q["invalid_ds"] += int(ds_date.isna().sum())
        q["invalid_accept_time"] += int(accept.isna().sum())
        q["invalid_delivery_time"] += int(delivery.isna().sum())

        # 4. Kiểm tra thời gian nằm ngoài phạm vi công bố
        q["ds_outside_range"] += int(
            (
                ds_date.notna() &
                ((ds_date < START_DATE) |
                 (ds_date > END_DATE))
            ).sum()
        )

        q["accept_outside_range"] += int(
            (
                accept.notna() &
                ((accept.dt.normalize() < START_DATE) |
                 (accept.dt.normalize() > END_DATE))
            ).sum()
        )

        q["delivery_outside_range"] += int(
            (
                delivery.notna() &
                ((delivery.dt.normalize() < START_DATE) |
                 (delivery.dt.normalize() > END_DATE))
            ).sum()
        )

        # 5. Kiểm tra thứ tự thời gian
        valid_pair = accept.notna() & delivery.notna()

        duration_hours = (
            delivery - accept
        ).dt.total_seconds() / 3600

        q["valid_time_pairs"] += int(valid_pair.sum())

        q["delivery_before_accept"] += int(
            (valid_pair & (duration_hours < 0)).sum()
        )

        # Chỉ đánh dấu để kiểm tra, chưa kết luận là lỗi
        q["duration_over_24h"] += int(
            (valid_pair & (duration_hours > 24)).sum()
        )

        # 6. Kiểm tra ds trùng với ngày nào
        accept_day = accept.dt.normalize()
        delivery_day = delivery.dt.normalize()

        valid_ds_accept = ds_date.notna() & accept.notna()
        valid_ds_delivery = ds_date.notna() & delivery.notna()

        q["ds_accept_pairs"] += int(valid_ds_accept.sum())
        q["ds_delivery_pairs"] += int(valid_ds_delivery.sum())

        q["ds_matches_accept"] += int(
            (valid_ds_accept & (ds_date == accept_day)).sum()
        )

        q["ds_matches_delivery"] += int(
            (valid_ds_delivery & (ds_date == delivery_day)).sum()
        )

        # 7. Khoảng thời gian thực tế quan sát được
        for series, name in [
            (accept, "accept"),
            (delivery, "delivery")
        ]:
            valid = series.dropna()

            if valid.empty:
                continue

            current_min = valid.min()
            current_max = valid.max()

            if name == "accept":
                accept_min = (
                    current_min if accept_min is None
                    else min(accept_min, current_min)
                )
                accept_max = (
                    current_max if accept_max is None
                    else max(accept_max, current_max)
                )
            else:
                delivery_min = (
                    current_min if delivery_min is None
                    else min(delivery_min, current_min)
                )
                delivery_max = (
                    current_max if delivery_max is None
                    else max(delivery_max, current_max)
                )

        # 8. Đếm số đơn theo region và ds
        ds_text = ds_date.dt.strftime("%Y-%m-%d")

        tmp_ds = pd.DataFrame({
            "region": region,
            "date": ds_text
        }).dropna()

        for (r, d), n in (
            tmp_ds.groupby(["region", "date"]).size().items()
        ):
            daily_counts[
                (city_code, str(r), str(d))
            ] += int(n)

        days_seen.update(
            ds_text.dropna().unique().tolist()
        )

        # 9. Đếm số đơn theo ngày hoàn tất giao hàng
        completed_text = delivery.dt.strftime("%Y-%m-%d")

        tmp_completed = pd.DataFrame({
            "region": region,
            "date": completed_text
        }).dropna()

        for (r, d), n in (
            tmp_completed.groupby(
                ["region", "date"]
            ).size().items()
        ):
            daily_delivery_counts[
                (city_code, str(r), str(d))
            ] += int(n)

        # 10. Đếm số đơn theo AOI trong từng region
        tmp_aoi = pd.DataFrame({
            "region": region,
            "aoi": aoi
        }).dropna()

        for (r, a), n in (
            tmp_aoi.groupby(["region", "aoi"]).size().items()
        ):
            aoi_counts[
                (city_code, str(r), str(a))
            ] += int(n)

        for r, a in (
            tmp_aoi[["region", "aoi"]]
            .drop_duplicates()
            .itertuples(index=False, name=None)
        ):
            aoi_regions[(city_code, str(a))].add(str(r))

        # 11. Đếm số đơn theo giờ hoàn tất giao hàng
        tmp_hour = pd.DataFrame({
            "region": region,
            "hour": delivery.dt.hour
        }).dropna()

        for (r, h), n in (
            tmp_hour.groupby(["region", "hour"]).size().items()
        ):
            hour_counts[
                (city_code, str(r), int(h))
            ] += int(n)

        # 12. Kiểm tra tọa độ
        lng = pd.to_numeric(ch["lng"], errors="coerce")
        lat = pd.to_numeric(ch["lat"], errors="coerce")

        finite_xy = np.isfinite(lng) & np.isfinite(lat)

        q["invalid_coordinates"] += int(
            (~finite_xy).sum()
        )

        q["coordinates_in_degree_range"] += int(
            (
                finite_xy &
                lng.between(-180, 180) &
                lat.between(-90, 90)
            ).sum()
        )

    # 13. Tổng hợp kết quả từng thành phố
    quality_rows.append({
        "city_code": city_code,
        "city_name": ", ".join(sorted(city_names)),
        "total_rows": q["total_rows"],
        "unique_regions": len(city_region_ids[city_code]),
        "unique_ds_days": len(days_seen),
        "first_ds": min(days_seen) if days_seen else None,
        "last_ds": max(days_seen) if days_seen else None,
        "first_accept": str(accept_min),
        "last_accept": str(accept_max),
        "first_delivery": str(delivery_min),
        "last_delivery": str(delivery_max),
        "duplicate_order_ids": q["duplicate_order_ids"],
        "invalid_ds": q["invalid_ds"],
        "invalid_accept_time": q["invalid_accept_time"],
        "invalid_delivery_time": q["invalid_delivery_time"],
        "ds_outside_range": q["ds_outside_range"],
        "accept_outside_range": q["accept_outside_range"],
        "delivery_outside_range": q["delivery_outside_range"],
        "delivery_before_accept": q["delivery_before_accept"],
        "duration_over_24h": q["duration_over_24h"],
        "invalid_coordinates": q["invalid_coordinates"],
        "ds_accept_match_pct": round(
            100 * q["ds_matches_accept"] /
            q["ds_accept_pairs"], 3
        ) if q["ds_accept_pairs"] else np.nan,
        "ds_delivery_match_pct": round(
            100 * q["ds_matches_delivery"] /
            q["ds_delivery_pairs"], 3
        ) if q["ds_delivery_pairs"] else np.nan
    })

quality_df = pd.DataFrame(quality_rows)

display(quality_df)

quality_df.to_csv(
    OUT_DIR / "04_quality_region_time_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

# 14. Kiểm tra mã region xuất hiện ở nhiều thành phố
region_to_cities = defaultdict(set)

for city, regions in city_region_ids.items():
    for r in regions:
        region_to_cities[r].add(city)

cross_city_regions = pd.DataFrame([
    {
        "region_id": r,
        "n_cities": len(cities),
        "cities": ", ".join(sorted(cities))
    }
    for r, cities in region_to_cities.items()
    if len(cities) > 1
], columns=["region_id", "n_cities", "cities"])

cross_city_regions.to_csv(
    OUT_DIR / "05_cross_city_region_ids.csv",
    index=False,
    encoding="utf-8-sig"
)

# 15. Kiểm tra AOI được ghi nhận ở nhiều region
# trong cùng một thành phố
aoi_conflicts = pd.DataFrame([
    {
        "city_code": city,
        "aoi_id": aoi,
        "n_regions": len(regions),
        "region_ids": ", ".join(sorted(regions))
    }
    for (city, aoi), regions in aoi_regions.items()
    if len(regions) > 1
], columns=[
    "city_code", "aoi_id", "n_regions", "region_ids"
])

aoi_conflicts.to_csv(
    OUT_DIR / "06_aoi_multiple_regions.csv",
    index=False,
    encoding="utf-8-sig"
)

# 16. Lưu riêng hai cách thống kê theo ngày
daily_ds_df = pd.DataFrame(
    [
        (city, region, date, n)
        for (city, region, date), n in daily_counts.items()
    ],
    columns=["city", "region_id", "ds_date", "n_orders"]
)

daily_delivery_df = pd.DataFrame(
    [
        (city, region, date, n)
        for (city, region, date), n in
        daily_delivery_counts.items()
    ],
    columns=[
        "city", "region_id",
        "completed_date", "n_orders"
    ]
)

daily_ds_df.to_csv(
    OUT_DIR / "07_daily_counts_by_ds.csv",
    index=False,
    encoding="utf-8-sig"
)

daily_delivery_df.to_csv(
    OUT_DIR / "08_daily_counts_by_completion_date.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nKIỂM TRA HOÀN TẤT")
print("Phạm vi công bố: 01/05/2022 - 31/10/2022")
print("Tổng bản ghi:", f"{quality_df.total_rows.sum():,}")
print("Số mã region trùng giữa các thành phố:",
      len(cross_city_regions))
print("Số AOI xuất hiện ở nhiều region cùng thành phố:",
      len(aoi_conflicts))

print("\nCác biến có thể dùng cho Cell 6:")
print("daily_counts, daily_delivery_counts,")
print("aoi_counts, hour_counts, quality_df")

# CELL 5A - KIỂM TRA CHI TIẾT NGOẠI LỆ THỜI GIAN
# Dùng lại FILES và OUT_DIR từ các cell trước

import pandas as pd
from collections import Counter

START = pd.Timestamp("2022-05-01")
END = pd.Timestamp("2022-10-31")

counts = Counter()
examples = []
MAX_EXAMPLES = 15

for city, path in FILES.items():
    print("Checking:", city)

    for ch in pd.read_csv(
        path,
        usecols=[
            "region_id", "ds",
            "accept_time", "delivery_time"
        ],
        dtype={"region_id": "string", "ds": "string"},
        chunksize=200_000
    ):
        ds_text = ch["ds"].str.zfill(4)

        ds_date = pd.to_datetime(
            "2022-" + ds_text.str[:2] + "-" +
            ds_text.str[2:4],
            format="%Y-%m-%d",
            errors="coerce"
        )

        accept = pd.to_datetime(
            "2022-" + ch["accept_time"].astype("string"),
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        delivery = pd.to_datetime(
            "2022-" + ch["delivery_time"].astype("string"),
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        valid_pair = accept.notna() & delivery.notna()

        flags = {
            "delivery_before_accept": (
                valid_pair & (delivery < accept)
            ),
            "accept_before_may": (
                accept.notna() &
                (accept.dt.normalize() < START)
            ),
            "accept_after_october": (
                accept.notna() &
                (accept.dt.normalize() > END)
            ),
            "delivery_before_may": (
                delivery.notna() &
                (delivery.dt.normalize() < START)
            ),
            "delivery_after_october": (
                delivery.notna() &
                (delivery.dt.normalize() > END)
            ),
            "duration_over_24h": (
                valid_pair &
                ((delivery - accept) > pd.Timedelta(hours=24))
            )
        }

        for label, mask in flags.items():
            counts[(city, label)] += int(mask.sum())

            # Chỉ lưu một số dòng để kiểm tra,
            # không xuất ID đơn hàng hay tọa độ cá nhân.
            existing = sum(
                1 for e in examples
                if e["city"] == city and e["issue"] == label
            )

            remaining = MAX_EXAMPLES - existing

            if remaining > 0 and mask.any():
                selected = ch.loc[mask].head(remaining)

                for idx, row in selected.iterrows():
                    examples.append({
                        "city": city,
                        "issue": label,
                        "region_id": row["region_id"],
                        "ds": row["ds"],
                        "accept_time": row["accept_time"],
                        "delivery_time": row["delivery_time"]
                    })

summary = pd.DataFrame(
    [
        {"city": c, "issue": issue, "count": n}
        for (c, issue), n in counts.items()
    ]
)

summary_pivot = summary.pivot_table(
    index="city",
    columns="issue",
    values="count",
    aggfunc="sum",
    fill_value=0
).reset_index()

examples_df = pd.DataFrame(examples)

display(summary_pivot)
display(examples_df)

summary_pivot.to_csv(
    OUT_DIR / "12_time_anomaly_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

examples_df.to_csv(
    OUT_DIR / "13_time_anomaly_examples.csv",
    index=False,
    encoding="utf-8-sig"
)

# CELL 6 - REGION FEATURES & ELIGIBILITY AUDIT
# Chưa chạy clustering

import pandas as pd
import numpy as np

# 1. Số đơn theo region/ngày ds
daily_df = pd.DataFrame(
    [
        (city, region, date, n)
        for (city, region, date), n in daily_counts.items()
    ],
    columns=["city", "region_id", "date_ds", "orders"]
)

keys = ["city", "region_id"]

# 2. Thống kê khối lượng và biến động
volume = (
    daily_df.groupby(keys)
    .agg(
        observed_days=("date_ds", "nunique"),
        total_orders=("orders", "sum"),
        avg_daily_volume=("orders", "mean"),
        median_daily_volume=("orders", "median"),
        sd_daily_volume=("orders", "std"),
        p95_daily_volume=(
            "orders", lambda s: s.quantile(0.95)
        )
    )
    .reset_index()
)

volume["volume_cv"] = (
    volume["sd_daily_volume"] /
    volume["avg_daily_volume"]
)

# Kỳ nghiên cứu từ 01/05 đến 31/10/2022
calendar_days = len(pd.date_range(
    "2022-05-01", "2022-10-31"
))

volume["calendar_days"] = calendar_days

volume["recorded_day_coverage"] = (
    volume["observed_days"] / calendar_days
)

# 3. Chỉ số tập trung AOI
aoi_df = pd.DataFrame(
    [
        (city, region, aoi, n)
        for (city, region, aoi), n in aoi_counts.items()
    ],
    columns=["city", "region_id", "aoi_id", "orders"]
)

aoi_df["share"] = (
    aoi_df["orders"] /
    aoi_df.groupby(keys)["orders"].transform("sum")
)

aoi_df["share_sq"] = aoi_df["share"] ** 2

aoi_stats = (
    aoi_df.groupby(keys)
    .agg(
        n_aoi=("aoi_id", "nunique"),
        aoi_hhi=("share_sq", "sum"),
        top_aoi_share=("share", "max")
    )
    .reset_index()
)

# 4. Phân bố giờ hoàn tất giao hàng
hour_df = pd.DataFrame(
    [
        (city, region, hour, n)
        for (city, region, hour), n in hour_counts.items()
    ],
    columns=["city", "region_id", "hour", "orders"]
)

hour_df["share"] = (
    hour_df["orders"] /
    hour_df.groupby(keys)["orders"].transform("sum")
)

hour_df["entropy_part"] = -(
    hour_df["share"] * np.log(hour_df["share"])
)

hour_stats = (
    hour_df.groupby(keys)
    .agg(
        peak_hour_share=("share", "max"),
        hourly_entropy=("entropy_part", "sum")
    )
    .reset_index()
)

hour_stats["hourly_entropy_normalized"] = (
    hour_stats["hourly_entropy"] / np.log(24)
)

# 5. Ghép các đặc trưng
features_df = (
    volume
    .merge(aoi_stats, on=keys, how="left")
    .merge(hour_stats, on=keys, how="left")
)

features_df["region_key"] = (
    features_df["city"] + "::" +
    features_df["region_id"].astype(str)
)

# 6. Thống kê số khu vực
region_summary = (
    features_df.groupby("city")
    .agg(
        n_regions=("region_id", "nunique"),
        total_orders=("total_orders", "sum"),
        min_observed_days=("observed_days", "min"),
        median_observed_days=("observed_days", "median"),
        max_observed_days=("observed_days", "max"),
        median_coverage=("recorded_day_coverage", "median")
    )
    .reset_index()
)

# 7. Đánh giá độ nhạy theo nhiều ngưỡng
# Chỉ là ngưỡng thử, chưa phải quy tắc loại mẫu cuối cùng
screening = []

for min_days in [30, 60, 90, 120, 150, 180]:
    for min_orders in [100, 500, 1000]:

        mask = (
            (features_df["observed_days"] >= min_days) &
            (features_df["total_orders"] >= min_orders)
        )

        screening.append({
            "min_days": min_days,
            "min_orders": min_orders,
            "eligible_regions": int(mask.sum()),
            "total_regions": len(features_df)
        })

screening_df = pd.DataFrame(screening)

# 8. Xuất bằng chứng
features_df.to_csv(
    OUT_DIR / "14_region_candidate_features.csv",
    index=False, encoding="utf-8-sig"
)

region_summary.to_csv(
    OUT_DIR / "15_region_coverage_summary.csv",
    index=False, encoding="utf-8-sig"
)

screening_df.to_csv(
    OUT_DIR / "16_region_eligibility_sensitivity.csv",
    index=False, encoding="utf-8-sig"
)

print("Total region keys:", len(features_df))

print("\n1. REGION SUMMARY")
display(region_summary)

print("\n2. FEATURE DESCRIPTIVE STATISTICS")
display(
    features_df[
        [
            "avg_daily_volume",
            "volume_cv",
            "recorded_day_coverage",
            "aoi_hhi",
            "peak_hour_share",
            "hourly_entropy_normalized"
        ]
    ].describe()
)

print("\n3. ELIGIBILITY SENSITIVITY")
display(screening_df)

print("\n4. LOWEST DAY COVERAGE")
display(
    features_df.sort_values(
        "recorded_day_coverage"
    )[
        [
            "region_key", "observed_days",
            "calendar_days", "recorded_day_coverage",
            "total_orders"
        ]
    ].head(20)
)

# ======================================================
# CELL 7 - REGION TEMPORAL COVERAGE AUDIT
# Mục đích: kiểm tra độ phủ theo tháng, khoảng trống
# và số khu vực đạt các ngưỡng sàng lọc.
# Chưa chạy clustering.
# ======================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = [
    (city, region, day, n)
    for (city, region, day), n in daily_counts.items()
]

audit_daily = pd.DataFrame(
    rows,
    columns=["city", "region_id", "date", "orders"]
)

audit_daily["date"] = pd.to_datetime(
    audit_daily["date"],
    format="%Y-%m-%d",
    errors="raise"
)

audit_daily["month"] = (
    audit_daily["date"].dt.strftime("%Y-%m")
)

keys = ["city", "region_id"]

# 1. Hồ sơ dữ liệu theo region
region_temporal = (
    audit_daily.groupby(keys)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observed_days=("date", "nunique"),
        total_orders=("orders", "sum"),
        active_months=("month", "nunique")
    )
    .reset_index()
)

region_temporal["span_days"] = (
    region_temporal["last_date"] -
    region_temporal["first_date"]
).dt.days + 1

region_temporal["coverage_within_span"] = (
    region_temporal["observed_days"] /
    region_temporal["span_days"]
)

region_temporal["coverage_184_days"] = (
    region_temporal["observed_days"] / 184
)

# 2. Khoảng trống liên tiếp lớn nhất bên trong
# khoảng từ ngày ghi nhận đầu đến ngày ghi nhận cuối
gap_results = []

for (city, region), group in audit_daily.groupby(keys):

    dates = (
        group["date"]
        .drop_duplicates()
        .sort_values()
    )

    gaps = (
        dates.diff().dt.days - 1
    ).clip(lower=0)

    max_gap = (
        int(gaps.max())
        if gaps.notna().any()
        else 0
    )

    gap_results.append({
        "city": city,
        "region_id": region,
        "max_internal_gap_days": max_gap
    })

gap_df = pd.DataFrame(gap_results)

region_temporal = region_temporal.merge(
    gap_df, on=keys, how="left"
)

# 3. Số ngày có dữ liệu trong từng tháng
monthly = (
    audit_daily.groupby(keys + ["month"])
    .agg(
        days_with_records=("date", "nunique"),
        packages=("orders", "sum")
    )
    .reset_index()
)

monthly_days = (
    monthly.pivot_table(
        index=keys,
        columns="month",
        values="days_with_records",
        fill_value=0,
        aggfunc="sum"
    )
    .reindex(
        columns=[
            "2022-05", "2022-06", "2022-07",
            "2022-08", "2022-09", "2022-10"
        ],
        fill_value=0
    )
    .reset_index()
)

monthly_days.columns = [
    "city", "region_id",
    "days_2022_05", "days_2022_06",
    "days_2022_07", "days_2022_08",
    "days_2022_09", "days_2022_10"
]

region_temporal = region_temporal.merge(
    monthly_days, on=keys, how="left"
)

# 4. Các phương án chọn mẫu
for threshold in [90, 120, 150, 180]:
    region_temporal[f"eligible_{threshold}d"] = (
        region_temporal["observed_days"] >= threshold
    )

# Phương án làm việc: >=120 ngày
region_temporal["candidate_main_sample"] = (
    region_temporal["eligible_120d"]
)

# 5. Kiểm tra thành phần mẫu theo thành phố
city_screening = (
    region_temporal.groupby("city")
    .agg(
        total_regions=("region_id", "nunique"),
        eligible_90=("eligible_90d", "sum"),
        eligible_120=("eligible_120d", "sum"),
        eligible_150=("eligible_150d", "sum"),
        eligible_180=("eligible_180d", "sum"),
        regions_with_6_months=("active_months",
                               lambda s: int((s == 6).sum())),
        median_active_months=("active_months", "median"),
        median_coverage=("coverage_184_days", "median"),
        max_internal_gap=("max_internal_gap_days", "max")
    )
    .reset_index()
)

# 6. Các khu vực không đạt ngưỡng 120 ngày
excluded_120 = region_temporal.loc[
    ~region_temporal["eligible_120d"]
].sort_values(["city", "observed_days"])

# 7. Lưu kết quả
region_temporal.to_csv(
    OUT_DIR / "17_region_temporal_coverage.csv",
    index=False, encoding="utf-8-sig"
)

monthly.to_csv(
    OUT_DIR / "18_region_monthly_coverage.csv",
    index=False, encoding="utf-8-sig"
)

city_screening.to_csv(
    OUT_DIR / "19_city_sample_screening.csv",
    index=False, encoding="utf-8-sig"
)

excluded_120.to_csv(
    OUT_DIR / "20_regions_below_120_days.csv",
    index=False, encoding="utf-8-sig"
)

print("A. SỐ KHU VỰC ĐỦ ĐIỀU KIỆN THEO THÀNH PHỐ")
display(city_screening)

print("\nB. CÁC KHU VỰC DƯỚI 120 NGÀY")
display(
    excluded_120[
        [
            "city", "region_id", "observed_days",
            "active_months", "total_orders",
            "max_internal_gap_days"
        ]
    ]
)

print("\nC. SỐ KHU VỰC CÓ DỮ LIỆU ĐỦ 6 THÁNG")
print(int((region_temporal["active_months"] == 6).sum()))

print("\nD. KHOẢNG TRỐNG DỮ LIỆU LỚN NHẤT")
display(
    region_temporal.sort_values(
        "max_internal_gap_days", ascending=False
    ).head(15)[
        [
            "city", "region_id", "observed_days",
            "active_months", "max_internal_gap_days",
            "coverage_within_span"
        ]
    ]
)

# ======================================================
# CELL 8 - AOI / REGION CONSISTENCY AUDIT
# Kiểm tra AOI nhiều region:
# (1) Trong cả kỳ
# (2) Trong cùng một ngày
# ======================================================

import pandas as pd

aoi_audit_rows = []
same_day_examples = []

for city_code, path in FILES.items():

    print("Checking AOI-region:", city_code)

    # Chỉ đọc các cột định danh
    parts = []

    for chunk in pd.read_csv(
        path,
        usecols=["aoi_id", "region_id", "ds"],
        chunksize=200_000
    ):
        # Giữ duy nhất mỗi tổ hợp AOI-region-ngày
        parts.append(
            chunk.drop_duplicates()
        )

    mapping = (
        pd.concat(parts, ignore_index=True)
        .drop_duplicates()
    )

    # Số region của mỗi AOI trong cả kỳ
    region_per_aoi = (
        mapping.groupby("aoi_id")["region_id"]
        .nunique()
    )

    multiple_lifetime = set(
        region_per_aoi[
            region_per_aoi > 1
        ].index
    )

    # Số region của mỗi AOI trong cùng một ngày
    region_per_aoi_day = (
        mapping.groupby(["aoi_id", "ds"])["region_id"]
        .nunique()
    )

    same_day_multi = (
        region_per_aoi_day[
            region_per_aoi_day > 1
        ]
    )

    same_day_aoi_ids = set(
        same_day_multi.index.get_level_values("aoi_id")
    )

    # AOI nhiều region trong cả kỳ,
    # nhưng không xuất hiện nhiều region cùng ngày
    multi_only_across_days = (
        multiple_lifetime - same_day_aoi_ids
    )

    aoi_audit_rows.append({
        "city": city_code,
        "unique_aoi_ids": int(mapping["aoi_id"].nunique()),
        "aoi_multiple_regions_overall": len(
            multiple_lifetime
        ),
        "aoi_multiple_regions_same_day": len(
            same_day_aoi_ids
        ),
        "aoi_day_cases_multiple_regions": int(
            len(same_day_multi)
        ),
        "aoi_multiple_regions_only_across_days": len(
            multi_only_across_days
        )
    })

    # Lưu ví dụ cấp AOI-ngày, không chứa ID đơn
    if len(same_day_multi):
        sample = (
            same_day_multi.reset_index(
                name="number_of_regions"
            )
            .sort_values(
                "number_of_regions", ascending=False
            )
            .head(20)
        )

        sample["city"] = city_code
        same_day_examples.append(sample)

aoi_stability_df = pd.DataFrame(aoi_audit_rows)

if same_day_examples:
    aoi_examples_df = pd.concat(
        same_day_examples, ignore_index=True
    )
else:
    aoi_examples_df = pd.DataFrame(
        columns=[
            "aoi_id", "ds",
            "number_of_regions", "city"
        ]
    )

aoi_stability_df.to_csv(
    OUT_DIR / "21_aoi_region_consistency.csv",
    index=False, encoding="utf-8-sig"
)

aoi_examples_df.to_csv(
    OUT_DIR / "22_aoi_same_day_examples.csv",
    index=False, encoding="utf-8-sig"
)

print("\nAOI-REGION CONSISTENCY")
display(aoi_stability_df)

print("\nTOTAL AOI IDS IN MULTIPLE REGIONS:")
print(
    aoi_stability_df[
        "aoi_multiple_regions_overall"
    ].sum()
)

print("\nSAME-DAY CASES:")
display(aoi_examples_df.head(20))

# CELL 9 - REGION ELIGIBILITY, COMBINED CRITERIA
# Chạy sau Cell 7 và Cell 8. Chưa thực hiện clustering.

import pandas as pd
import numpy as np

selection = region_temporal.copy()

month_cols = [
    "days_2022_05", "days_2022_06", "days_2022_07",
    "days_2022_08", "days_2022_09", "days_2022_10"
]

# Kiểm tra các cột bắt buộc
required = [
    "city", "region_id", "observed_days",
    "active_months", "max_internal_gap_days"
] + month_cols

missing = [c for c in required if c not in selection.columns]
if missing:
    raise ValueError(f"Thiếu cột từ Cell 7: {missing}")

# Điều kiện 1: ít nhất 120 ngày có bản ghi
selection["days_120"] = (
    selection["observed_days"] >= 120
)

# Điều kiện 2: có bản ghi trong cả 6 tháng
selection["all_6_months"] = (
    selection["active_months"] == 6
)

# Phương án mẫu chính - cần GVHD thống nhất
selection["candidate_main"] = (
    selection["days_120"] &
    selection["all_6_months"]
)

# Độ nhạy của số ngày trong từng tháng
selection["min_days_in_month"] = selection[
    month_cols
].min(axis=1)

# Các ngưỡng 5 và 10 ngày/tháng chỉ để kiểm tra độ nhạy
selection["candidate_min5_each_month"] = (
    selection["candidate_main"] &
    (selection["min_days_in_month"] >= 5)
)

selection["candidate_min10_each_month"] = (
    selection["candidate_main"] &
    (selection["min_days_in_month"] >= 10)
)

# Kiểm tra thêm ảnh hưởng của khoảng trống nội kỳ
selection["candidate_maxgap14"] = (
    selection["candidate_main"] &
    (selection["max_internal_gap_days"] <= 14)
)

summary = (
    selection.groupby("city")
    .agg(
        total_regions=("region_id", "size"),
        days_120=("days_120", "sum"),
        all_6_months=("all_6_months", "sum"),
        candidate_main=("candidate_main", "sum"),
        min5_each_month=("candidate_min5_each_month", "sum"),
        min10_each_month=("candidate_min10_each_month", "sum"),
        maxgap14=("candidate_maxgap14", "sum")
    )
    .reset_index()
)

# Thêm dòng tổng
total = {"city": "TOTAL"}
for c in summary.columns:
    if c != "city":
        total[c] = int(summary[c].sum())

summary = pd.concat(
    [summary, pd.DataFrame([total])],
    ignore_index=True
)

print("BẢNG SÀNG LỌC TỔNG HỢP")
display(summary)

print("\nCÁC REGION KHÔNG ĐẠT ĐỒNG THỜI 2 ĐIỀU KIỆN:")
display(
    selection.loc[
        ~selection["candidate_main"],
        ["city", "region_id", "observed_days",
         "active_months", "min_days_in_month",
         "max_internal_gap_days"]
    ].sort_values(["city", "observed_days"])
)

# Lưu bằng chứng đã thực hành
selection.to_csv(
    OUT_DIR / "23_region_final_screening.csv",
    index=False, encoding="utf-8-sig"
)

summary.to_csv(
    OUT_DIR / "24_sample_comparison_by_city.csv",
    index=False, encoding="utf-8-sig"
)

# Kiểm tra kết quả với các thống kê đã chạy trước đó
assert int(selection["days_120"].sum()) == 163
assert int(selection["all_6_months"].sum()) == 160

print("\nĐã xác nhận lại:")
print(">=120 ngày:", int(selection["days_120"].sum()))
print("Đủ 6 tháng:", int(selection["all_6_months"].sum()))
print("Đạt đồng thời:", int(selection["candidate_main"].sum()))

# =====================================================
# CELL 10 - SPATIAL FEATURES & MONTHLY CENTROID SHIFT
# LaDe-D | Không chạy clustering
# =====================================================

from collections import defaultdict
import pandas as pd
import numpy as np

CHUNK_SIZE = 200_000

def haversine_km(lon1, lat1, lon2, lat2):
    """Khoảng cách hình học trên mặt cầu, km."""
    R = 6371.0088

    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    a = (
        np.sin((lat2 - lat1) / 2) ** 2 +
        np.cos(lat1) * np.cos(lat2) *
        np.sin((lon2 - lon1) / 2) ** 2
    )

    a = np.clip(a, 0, 1)
    return 2 * R * np.arcsin(np.sqrt(a))


# 1. Tính tâm tọa độ cho từng region
global_sum = defaultdict(
    lambda: np.zeros(3, dtype=float)
)
monthly_sum = defaultdict(
    lambda: np.zeros(3, dtype=float)
)
invalid_xy_counts = defaultdict(int)

usecols = ["region_id", "ds", "lng", "lat"]

for city, path in FILES.items():
    print("PASS 1:", city)

    for ch in pd.read_csv(
        path,
        usecols=usecols,
        dtype={"region_id": "string"},
        chunksize=CHUNK_SIZE
    ):
        lon = pd.to_numeric(ch["lng"], errors="coerce")
        lat = pd.to_numeric(ch["lat"], errors="coerce")
        month = pd.to_numeric(
            ch["ds"], errors="coerce"
        ).floordiv(100)

        valid = (
            lon.between(-180, 180) &
            lat.between(-90, 90) &
            np.isfinite(lon) &
            np.isfinite(lat) &
            ch["region_id"].notna() &
            month.between(5, 10)
        )

        invalid_xy_counts[city] += int((~valid).sum())

        z = pd.DataFrame({
            "region": ch.loc[
                valid, "region_id"
            ].astype(str),
            "month": month.loc[valid].astype(int),
            "lon": lon.loc[valid].astype(float),
            "lat": lat.loc[valid].astype(float)
        })

        g = z.groupby("region").agg(
            n=("lon", "size"),
            sx=("lon", "sum"),
            sy=("lat", "sum")
        )

        for region, v in g.iterrows():
            global_sum[(city, region)] += np.array(
                [v["n"], v["sx"], v["sy"]],
                dtype=float
            )

        gm = z.groupby(["region", "month"]).agg(
            n=("lon", "size"),
            sx=("lon", "sum"),
            sy=("lat", "sum")
        )

        for (region, m), v in gm.iterrows():
            monthly_sum[(city, region, int(m))] += (
                np.array(
                    [v["n"], v["sx"], v["sy"]],
                    dtype=float
                )
            )

# Tọa độ tâm phân bố của từng region
centroids = {
    key: (v[1] / v[0], v[2] / v[0])
    for key, v in global_sum.items()
    if v[0] > 0
}

print("Số region có tâm tọa độ:", len(centroids))
print("Số dòng không đạt kiểm tra sơ bộ:",
      dict(invalid_xy_counts))


# 2. Tính khoảng cách từng điểm giao đến tâm
distance_parts = defaultdict(list)

for city, path in FILES.items():
    print("PASS 2:", city)

    city_centroids = {
        r: coord
        for (c, r), coord in centroids.items()
        if c == city
    }

    lon_lookup = {
        r: coord[0]
        for r, coord in city_centroids.items()
    }
    lat_lookup = {
        r: coord[1]
        for r, coord in city_centroids.items()
    }

    for ch in pd.read_csv(
        path,
        usecols=usecols,
        dtype={"region_id": "string"},
        chunksize=CHUNK_SIZE
    ):
        reg = ch["region_id"].astype("string")
        lon = pd.to_numeric(
            ch["lng"], errors="coerce"
        )
        lat = pd.to_numeric(
            ch["lat"], errors="coerce"
        )
        month = pd.to_numeric(
            ch["ds"], errors="coerce"
        ).floordiv(100)

        lon0 = reg.map(lon_lookup)
        lat0 = reg.map(lat_lookup)

        valid = (
            lon.between(-180, 180) &
            lat.between(-90, 90) &
            np.isfinite(lon) &
            np.isfinite(lat) &
            lon0.notna() &
            lat0.notna() &
            month.between(5, 10)
        )

        if not valid.any():
            continue

        distances = haversine_km(
            lon.loc[valid].to_numpy(float),
            lat.loc[valid].to_numpy(float),
            lon0.loc[valid].to_numpy(float),
            lat0.loc[valid].to_numpy(float)
        )

        temp = pd.DataFrame({
            "region": reg.loc[valid].astype(str),
            "dist_km": distances
        })

        for region, group in temp.groupby("region"):
            distance_parts[(city, region)].append(
                group["dist_km"].to_numpy(
                    dtype=np.float32
                )
            )


# 3. Tính chỉ số phân tán không gian
spatial_rows = []

for (city, region), parts in distance_parts.items():
    d = np.concatenate(parts)

    spatial_rows.append({
        "city": city,
        "region_id": region,
        "n_valid_coordinates": len(d),
        "median_distance_to_centroid_km": float(
            np.median(d)
        ),
        "p90_distance_to_centroid_km": float(
            np.quantile(d, 0.90)
        ),
        "mean_distance_to_centroid_km": float(
            d.mean()
        )
    })

spatial_df = pd.DataFrame(spatial_rows)


# 4. Kiểm tra tâm phân bố theo tháng
shift_rows = []

for (city, region, month), v in monthly_sum.items():

    if v[0] == 0:
        continue

    monthly_lon = v[1] / v[0]
    monthly_lat = v[2] / v[0]

    global_lon, global_lat = centroids[
        (city, region)
    ]

    shift = float(haversine_km(
        global_lon, global_lat,
        monthly_lon, monthly_lat
    ))

    shift_rows.append({
        "city": city,
        "region_id": region,
        "month": month,
        "n_points": int(v[0]),
        "monthly_centroid_shift_km": shift
    })

monthly_shift_df = pd.DataFrame(shift_rows)

shift_summary = (
    monthly_shift_df.groupby(
        ["city", "region_id"]
    )
    .agg(
        months_with_coordinates=("month", "nunique"),
        max_monthly_centroid_shift_km=(
            "monthly_centroid_shift_km", "max"
        ),
        median_monthly_centroid_shift_km=(
            "monthly_centroid_shift_km", "median"
        )
    )
    .reset_index()
)

# 5. Ghép chỉ số không gian với các biến đã tính
spatial_df = spatial_df.merge(
    shift_summary,
    on=["city", "region_id"],
    how="left"
)

# Lưu kết quả kiểm toán
spatial_df.to_csv(
    OUT_DIR / "25_region_spatial_features.csv",
    index=False, encoding="utf-8-sig"
)

monthly_shift_df.to_csv(
    OUT_DIR / "26_region_monthly_centroid_shift.csv",
    index=False, encoding="utf-8-sig"
)

# Ghép vào bảng đặc trưng đã tạo ở Cell 6
# Cell 6 đặt tên cột là city, region_id
features_spatial_df = features_df.merge(
    spatial_df,
    on=["city", "region_id"],
    how="left",
    validate="one_to_one"
)

features_spatial_df.to_csv(
    OUT_DIR / "27_candidate_features_with_spatial.csv",
    index=False, encoding="utf-8-sig"
)

print("\nA. MÔ TẢ BIẾN KHÔNG GIAN")
display(
    spatial_df[
        [
            "median_distance_to_centroid_km",
            "p90_distance_to_centroid_km",
            "max_monthly_centroid_shift_km"
        ]
    ].describe()
)

print("\nB. 15 REGION CÓ TÂM TỌA ĐỘ BIẾN ĐỘNG LỚN NHẤT")
display(
    spatial_df.sort_values(
        "max_monthly_centroid_shift_km",
        ascending=False
    ).head(15)
)

print("\nC. SỐ REGION CÓ CHỈ SỐ KHÔNG GIAN")
print(len(spatial_df))

# CELL 11 - EXPORT AUDIT EVIDENCE

from pathlib import Path
from google.colab import files
import shutil

OUT_DIR = Path(OUT_DIR)

print("Các tệp kết quả hiện có:")
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        print(p.name, f"{p.stat().st_size / 1024:.1f} KB")

zip_file = shutil.make_archive(
    "/content/LaDeD_Phase1_Evidence_20261101",
    "zip",
    root_dir=str(OUT_DIR)
)

print("\nĐã xuất:", zip_file)
files.download(zip_file)

# ======================================================
# CELL 12 - FINAL PHASE 1 SAMPLE & FEATURE VERIFICATION
# ======================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path("/content/LaDe_D_Research/results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Đọc trực tiếp kết quả của Cell 9 và Cell 10
selection = pd.read_csv(
    OUT_DIR / "23_region_final_screening.csv",
    dtype={"city": "string", "region_id": "string"}
)

features = pd.read_csv(
    OUT_DIR / "27_candidate_features_with_spatial.csv",
    dtype={"city": "string", "region_id": "string"}
)

KEYS = ["city", "region_id"]

# 1. Kiểm tra khóa khu vực
assert not selection.duplicated(KEYS).any(), \
    "Có khóa khu vực trùng trong bảng selection."

assert not features.duplicated(KEYS).any(), \
    "Có khóa khu vực trùng trong bảng features."

assert len(selection) == 172, \
    "Số region trong bảng selection khác 172."

assert len(features) == 172, \
    "Số region trong bảng features khác 172."

# 2. Ghép kết quả lựa chọn mẫu
selection_cols = KEYS + [
    "candidate_main",
    "candidate_min5_each_month",
    "candidate_min10_each_month",
    "candidate_maxgap14",
    "active_months",
    "max_internal_gap_days",
    "min_days_in_month"
]

merged = features.merge(
    selection[selection_cols],
    on=KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert merged["_merge"].eq("both").all(), \
    "Có region không ghép được giữa hai bảng."

merged = merged.drop(columns="_merge")

# 3. Chọn mẫu chính
main = merged.loc[
    merged["candidate_main"] == True
].copy()

assert len(main) == 160, \
    "Số region đạt đồng thời hai tiêu chí không bằng 160."

assert (main["observed_days"] >= 120).all()
assert (main["active_months"] == 6).all()

# 4. Kiểm tra biến
feature_cols = [
    "avg_daily_volume",
    "volume_cv",
    "aoi_hhi",
    "peak_hour_share",
    "hourly_entropy_normalized",
    "median_distance_to_centroid_km",
    "p90_distance_to_centroid_km",
    "max_monthly_centroid_shift_km"
]

missing_cols = [
    c for c in feature_cols if c not in main.columns
]

if missing_cols:
    raise ValueError(f"Thiếu biến: {missing_cols}")

# 5. Tính lại thống kê mô tả CHỈ trên 160 region
main_describe = (
    main[feature_cols]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .T
    .reset_index()
    .rename(columns={"index": "feature"})
)

# 6. Số region và số đơn còn lại theo thành phố
city_sample = (
    merged.groupby("city", as_index=False)
    .agg(
        regions_original=("region_id", "size"),
        orders_original=("total_orders", "sum")
    )
)

city_main = (
    main.groupby("city", as_index=False)
    .agg(
        regions_selected=("region_id", "size"),
        orders_selected=("total_orders", "sum")
    )
)

city_sample = city_sample.merge(
    city_main, on="city", how="left"
).fillna({
    "regions_selected": 0,
    "orders_selected": 0
})

city_sample["region_retention_pct"] = (
    100 * city_sample["regions_selected"] /
    city_sample["regions_original"]
)

city_sample["order_retention_pct"] = (
    100 * city_sample["orders_selected"] /
    city_sample["orders_original"]
)

total_row = pd.DataFrame([{
    "city": "TOTAL",
    "regions_original": city_sample["regions_original"].sum(),
    "orders_original": city_sample["orders_original"].sum(),
    "regions_selected": city_sample["regions_selected"].sum(),
    "orders_selected": city_sample["orders_selected"].sum(),
    "region_retention_pct": (
        100 * city_sample["regions_selected"].sum() /
        city_sample["regions_original"].sum()
    ),
    "order_retention_pct": (
        100 * city_sample["orders_selected"].sum() /
        city_sample["orders_original"].sum()
    )
}])

city_sample = pd.concat(
    [city_sample, total_row],
    ignore_index=True
)

# 7. Kiểm tra dữ liệu thiếu của bộ biến
feature_missing = pd.DataFrame({
    "feature": feature_cols,
    "missing_regions": [
        int(main[c].isna().sum())
        for c in feature_cols
    ],
    "available_regions": [
        int(main[c].notna().sum())
        for c in feature_cols
    ]
})

# 8. Thống kê theo thành phố để phát hiện city effect
city_feature_profile = (
    main.groupby("city")[feature_cols]
    .median()
    .reset_index()
)

# 9. Xuất các bảng bằng chứng
main.to_csv(
    OUT_DIR / "28_main_sample_160_regions.csv",
    index=False, encoding="utf-8-sig"
)

main_describe.to_csv(
    OUT_DIR / "29_main_sample_feature_statistics.csv",
    index=False, encoding="utf-8-sig"
)

city_sample.to_csv(
    OUT_DIR / "30_city_sample_retention.csv",
    index=False, encoding="utf-8-sig"
)

feature_missing.to_csv(
    OUT_DIR / "31_main_sample_feature_missingness.csv",
    index=False, encoding="utf-8-sig"
)

city_feature_profile.to_csv(
    OUT_DIR / "32_city_feature_profile.csv",
    index=False, encoding="utf-8-sig"
)

print("A. THÀNH PHẦN MẪU CHÍNH VÀ TỶ LỆ ĐƠN GIỮ LẠI")
display(city_sample)

print("\nB. THỐNG KÊ 8 BIẾN TRÊN ĐÚNG 160 KHU VỰC")
display(main_describe)

print("\nC. DỮ LIỆU THIẾU TRONG CÁC BIẾN")
display(feature_missing)

print("\nD. TRUNG VỊ CÁC BIẾN THEO THÀNH PHỐ")
display(city_feature_profile)

print("\nĐã lưu bảng kiểm tra cuối cùng.")

from pathlib import Path
from google.colab import files
import shutil

OUT_DIR = Path("/content/LaDe_D_Research/results")

final_zip = shutil.make_archive(
    "/content/LaDeD_Phase1_Final_Evidence_20261101",
    "zip",
    root_dir=str(OUT_DIR)
)

print("Số file bằng chứng:", len(list(OUT_DIR.glob("*"))))
print("ZIP cuối cùng:", final_zip)

files.download(final_zip)

# CELL 13 - FINAL AUDIT & FEATURE REDUNDANCY
# Không chạy clustering

from pathlib import Path
import pandas as pd
import numpy as np

OUT_DIR = Path("/content/LaDe_D_Research/results")

main = pd.read_csv(
    OUT_DIR / "28_main_sample_160_regions.csv"
)

quality = pd.read_csv(
    OUT_DIR / "04_quality_region_time_2022.csv"
)

features = [
    "avg_daily_volume",
    "volume_cv",
    "aoi_hhi",
    "peak_hour_share",
    "hourly_entropy_normalized",
    "median_distance_to_centroid_km",
    "p90_distance_to_centroid_km",
    "max_monthly_centroid_shift_km"
]

# 1. Xác nhận mẫu chính
assert len(main) == 160
assert main["total_orders"].sum() == 4_432_300
assert main[features].notna().all().all()
assert not main.duplicated(["city", "region_id"]).any()

print("PASS: 160 region, 4,432,300 records, 8 features complete.")

# 2. Kiểm tra kết quả trùng mã đơn
if "duplicate_order_ids" in quality.columns:
    print("\nDUPLICATE ORDER IDS BY CITY:")
    display(
        quality[
            ["city_code", "total_rows", "duplicate_order_ids"]
        ]
    )
    print(
        "Total duplicate order-ID occurrences:",
        int(quality["duplicate_order_ids"].sum())
    )
else:
    print("Chưa có cột duplicate_order_ids trong quality report.")

# 3. Tương quan Spearman giữa các biến
corr = main[features].corr(method="spearman")

print("\nSPEARMAN CORRELATION MATRIX:")
display(corr.round(3))

pairs = []

for i in range(len(features)):
    for j in range(i + 1, len(features)):
        r = corr.iloc[i, j]
        pairs.append({
            "feature_1": features[i],
            "feature_2": features[j],
            "spearman_r": r,
            "abs_r": abs(r)
        })

pairs_df = pd.DataFrame(pairs).sort_values(
    "abs_r", ascending=False
)

print("\nHIGHEST ABSOLUTE CORRELATIONS:")
display(pairs_df.head(10))

corr.to_csv(
    OUT_DIR / "33_feature_spearman_correlation.csv",
    encoding="utf-8-sig"
)

pairs_df.to_csv(
    OUT_DIR / "34_feature_correlation_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Lưu phiên bản kho dữ liệu được sử dụng
from huggingface_hub import HfApi

try:
    revision = HfApi().dataset_info("Cainiao-AI/LaDe").sha
    print("\nHugging Face source revision:", revision)
    (OUT_DIR / "35_source_revision.txt").write_text(
        f"repository: Cainiao-AI/LaDe\nrevision: {revision}\n",
        encoding="utf-8"
    )
except Exception as e:
    print("Không lấy được revision:", str(e))




sh 228069571 bytes
hz 286893611 bytes
cq 145068417 bytes
jl 4736342 bytes
yt 31532960 bytes
Download complete: 5 files

CITY: sh
Columns: ['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id', 'aoi_type', 'accept_time', 'accept_gps_time', 'accept_gps_lng', 'accept_gps_lat', 'delivery_time', 'delivery_gps_time', 'delivery_gps_lng', 'delivery_gps_lat', 'ds']
Data types:
order_id               int64
region_id              int64
city                  object
courier_id             int64
lng                  float64
lat                  float64
aoi_id                 int64
aoi_type               int64
accept_time           object
accept_gps_time       object
accept_gps_lng       float64
accept_gps_lat       float64
delivery_time         object
delivery_gps_time     object
delivery_gps_lng     float64
delivery_gps_lat     float64
ds                     int64


,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds
0,3158819,1,Shanghai,164,121.52128,31.06614,450,1,06-04 11:05:00,06-04 11:05:00,121.52281,31.10598,06-04 17:40:00,06-04 17:40:00,121.52407,31.06614,604
1,751342,1,Shanghai,164,121.52124,31.06687,450,1,06-04 11:18:00,06-04 11:18:00,121.52286,31.10599,06-04 15:06:00,06-04 15:06:00,121.52412,31.06618,604
2,3380476,1,Shanghai,164,121.52106,31.06731,450,1,06-03 10:13:00,06-03 10:13:00,121.52285,31.10591,06-03 15:11:00,06-03 15:11:00,121.52059,31.06672,603



CITY: hz
Columns: ['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id', 'aoi_type', 'accept_time', 'accept_gps_time', 'accept_gps_lng', 'accept_gps_lat', 'delivery_time', 'delivery_gps_time', 'delivery_gps_lng', 'delivery_gps_lat', 'ds']
Data types:
order_id               int64
region_id              int64
city                  object
courier_id             int64
lng                  float64
lat                  float64
aoi_id                 int64
aoi_type               int64
accept_time           object
accept_gps_time       object
accept_gps_lng       float64
accept_gps_lat       float64
delivery_time         object
delivery_gps_time     object
delivery_gps_lng     float64
delivery_gps_lat     float64
ds                     int64


,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds
0,583722,0,Hangzhou,175,120.17895,30.26401,749,1,10-30 09:20:00,10-30 09:20:00,120.20600,30.28657,10-30 10:30:00,10-30 10:30:00,120.17908,30.26392,1030
1,2819059,0,Hangzhou,175,120.17899,30.26336,749,1,10-31 09:47:00,10-31 09:47:00,120.20591,30.28651,10-31 10:40:00,10-31 10:40:00,120.17884,30.26360,1031
2,2879432,0,Hangzhou,175,120.17896,30.26404,749,1,10-22 10:11:00,10-22 10:11:00,120.20598,30.28668,10-22 15:03:00,10-22 15:03:00,120.17939,30.26395,1022



CITY: cq
Columns: ['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id', 'aoi_type', 'accept_time', 'accept_gps_time', 'accept_gps_lng', 'accept_gps_lat', 'delivery_time', 'delivery_gps_time', 'delivery_gps_lng', 'delivery_gps_lat', 'ds']
Data types:
order_id               int64
region_id              int64
city                  object
courier_id             int64
lng                  float64
lat                  float64
aoi_id                 int64
aoi_type               int64
accept_time           object
accept_gps_time       object
accept_gps_lng       float64
accept_gps_lat       float64
delivery_time         object
delivery_gps_time     object
delivery_gps_lng     float64
delivery_gps_lat     float64
ds                     int64


,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds
0,2031782,10,Chongqing,73,108.71571,30.90228,50,14,10-22 10:26:00,10-22 10:26:00,108.71826,30.95587,10-22 17:04:00,10-22 17:04:00,108.66361,30.96702,1022
1,4285071,10,Chongqing,3605,108.71639,30.90269,50,14,09-07 10:13:00,09-07 10:13:00,108.71791,30.95635,09-09 15:44:00,09-09 15:44:00,108.71644,30.90266,907
2,4056800,10,Chongqing,3605,108.71645,30.90259,50,14,06-26 09:49:00,06-26 09:49:00,108.71798,30.95635,06-27 16:03:00,06-27 16:03:00,108.71647,30.90251,626


... (59 outputs left out)

,avg_daily_volume,volume_cv,aoi_hhi,peak_hour_share,hourly_entropy_normalized,median_distance_to_centroid_km,p90_distance_to_centroid_km,max_monthly_centroid_shift_km
avg_daily_volume,1.000,-0.495,0.177,-0.077,0.101,-0.281,-0.381,-0.429
volume_cv,-0.495,1.000,-0.145,-0.295,0.312,0.137,0.182,0.336
aoi_hhi,0.177,-0.145,1.000,-0.125,0.074,-0.467,-0.490,-0.314
peak_hour_share,-0.077,-0.295,-0.125,1.000,-0.821,0.161,0.166,-0.012
hourly_entropy_normalized,0.101,0.312,0.074,-0.821,1.000,-0.123,-0.111,-0.030
median_distance_to_centroid_km,-0.281,0.137,-0.467,0.161,-0.123,1.000,0.942,0.745
p90_distance_to_centroid_km,-0.381,0.182,-0.490,0.166,-0.111,0.942,1.000,0.736
max_monthly_centroid_shift_km,-0.429,0.336,-0.314,-0.012,-0.030,0.745,0.736,1.000



HIGHEST ABSOLUTE CORRELATIONS:


,feature_1,feature_2,spearman_r,abs_r
25,median_distance_to_centroid_km,p90_distance_to_centroid_km,0.942136,0.942136
18,peak_hour_share,hourly_entropy_normalized,-0.821056,0.821056
26,median_distance_to_centroid_km,max_monthly_centroid_shift_km,0.744750,0.744750
27,p90_distance_to_centroid_km,max_monthly_centroid_shift_km,0.736312,0.736312
0,avg_daily_volume,volume_cv,-0.494823,0.494823
16,aoi_hhi,p90_distance_to_centroid_km,-0.490253,0.490253
15,aoi_hhi,median_distance_to_centroid_km,-0.467500,0.467500
6,avg_daily_volume,max_monthly_centroid_shift_km,-0.428630,0.428630
5,avg_daily_volume,p90_distance_to_centroid_km,-0.380713,0.380713
12,volume_cv,max_monthly_centroid_shift_km,0.335594,0.335594



Hugging Face source revision: be2cec02775cafc8d52230303f32134382bcc50b
